# Genomic Variant Classifier — Stage 2: Multi-Source Data Ingestion & Multi-Modal Modeling

**Author:** Monzia Moodie  
**Stage:** Production-scale pipeline — multi-database ingestion, feature engineering, tabular XGBoost classifier, and CNN branch for TCGA tumor image analysis  
**Databases integrated:** ClinVar · gnomAD · UniProt · GWAS Catalog · Open Targets Genetics · GTEx · ENCODE/SCREEN · CADD · REVEL · AlphaMissense · TCGA (GDC API)

---


## 1. Environment Setup & Dependency Validation

In [ ]:
# ── Install / verify all required packages ────────────────────────────────
import subprocess, sys

REQUIRED = [
    "pandas", "numpy", "requests", "tqdm",
    "scikit-learn", "xgboost", "lightgbm",
    "torch", "torchvision",
    "pyarrow", "pyvcf3",
    "matplotlib", "seaborn", "plotly",
    "pyspark",
]

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

missing = []
for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_").split("[")[0])
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing {len(missing)} missing packages: {missing}")
    for pkg in missing:
        install(pkg)
    print("All packages installed.")
else:
    print("All dependencies satisfied.")


In [ ]:
import os, json, time, hashlib, logging, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("GVC")

# ── Project directory structure ───────────────────────────────────────────────
BASE_DIR = Path("genomic_variant_classifier")
DIRS = {
    "raw":        BASE_DIR / "data" / "raw",
    "processed":  BASE_DIR / "data" / "processed",
    "features":   BASE_DIR / "data" / "features",
    "models":     BASE_DIR / "models",
    "images":     BASE_DIR / "data" / "images",
    "logs":       BASE_DIR / "logs",
    "reports":    BASE_DIR / "reports",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Project structure initialised:")
for k, v in DIRS.items():
    print(f"  {k:12s} → {v}")


## 2. Shared Utilities: Rate-Limited Requests, Retry Logic, Caching

In [ ]:
import time, functools
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CACHE = {}

def build_session(retries=5, backoff=1.5, timeout=30):
    """Requests session with exponential-backoff retry and sensible headers."""
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=Retry(
        total=retries,
        backoff_factor=backoff,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
    ))
    session.mount("https://", adapter)
    session.mount("http://",  adapter)
    session.headers.update({
        "User-Agent": "GenomicVariantClassifier/2.0 (research; contact@example.com)",
        "Accept": "application/json",
    })
    session.timeout = timeout
    return session

SESSION = build_session()

def cached_get(url, params=None, cache_key=None, ttl_hours=24):
    """GET with in-memory cache keyed by URL + params."""
    key = cache_key or hashlib.md5((url + str(params)).encode()).hexdigest()
    if key in CACHE:
        ts, data = CACHE[key]
        if (time.time() - ts) < ttl_hours * 3600:
            return data
    resp = SESSION.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    CACHE[key] = (time.time(), data)
    return data

def cached_post(url, payload, cache_key=None, ttl_hours=24):
    """POST with in-memory cache."""
    key = cache_key or hashlib.md5((url + json.dumps(payload, sort_keys=True)).encode()).hexdigest()
    if key in CACHE:
        ts, data = CACHE[key]
        if (time.time() - ts) < ttl_hours * 3600:
            return data
    resp = SESSION.post(url, json=payload)
    resp.raise_for_status()
    data = resp.json()
    CACHE[key] = (time.time(), data)
    return data

def safe_float(v, default=np.nan):
    try:
        return float(v)
    except (TypeError, ValueError):
        return default

print("Utilities loaded.")


## 3. ClinVar — Clinical Significance Labels (Supervised Targets)

ClinVar is the gold-standard source for pathogenicity labels. We query the NCBI E-utilities API
for a gene or variant of interest and parse the structured JSON response.


In [ ]:
CLINVAR_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
NCBI_API_KEY = os.environ.get("NCBI_API_KEY", "")   # set env var for higher rate limits

CLINVAR_SIG_MAP = {
    "Pathogenic": 2,
    "Likely pathogenic": 1,
    "Uncertain significance": 0,
    "Likely benign": -1,
    "Benign": -2,
}

def fetch_clinvar_variants(gene_symbol: str, max_records: int = 500) -> pd.DataFrame:
    """
    Query ClinVar for all variants associated with a gene symbol.
    Returns a DataFrame with rsID, clinical significance, review status, conditions.
    """
    log.info(f"ClinVar: searching {gene_symbol} (max {max_records})")
    # Step 1: esearch — get UIDs
    search_resp = SESSION.get(
        f"{CLINVAR_BASE}/esearch.fcgi",
        params={
            "db": "clinvar",
            "term": f"{gene_symbol}[Gene Name]",
            "retmax": max_records,
            "retmode": "json",
            "api_key": NCBI_API_KEY,
        }
    )
    search_resp.raise_for_status()
    uids = search_resp.json()["esearchresult"]["idlist"]
    if not uids:
        log.warning(f"ClinVar: no results for {gene_symbol}")
        return pd.DataFrame()

    log.info(f"ClinVar: found {len(uids)} variant UIDs for {gene_symbol}")

    # Step 2: efetch — get summaries in batches of 200
    records = []
    batch_size = 200
    for i in range(0, len(uids), batch_size):
        batch = uids[i:i + batch_size]
        fetch_resp = SESSION.get(
            f"{CLINVAR_BASE}/esummary.fcgi",
            params={
                "db": "clinvar",
                "id": ",".join(batch),
                "retmode": "json",
                "api_key": NCBI_API_KEY,
            }
        )
        fetch_resp.raise_for_status()
        result = fetch_resp.json().get("result", {})
        for uid in batch:
            v = result.get(uid, {})
            if not v or "error" in v:
                continue
            germline = v.get("germline_classification", {})
            sig = germline.get("description", "Unknown")
            records.append({
                "uid":              uid,
                "name":             v.get("title", ""),
                "gene_symbol":      gene_symbol,
                "clinical_sig":     sig,
                "sig_numeric":      CLINVAR_SIG_MAP.get(sig, np.nan),
                "review_status":    germline.get("review_status", ""),
                "last_evaluated":   germline.get("last_evaluated", ""),
                "num_submissions":  v.get("num_submissions", 0),
                "variation_type":   v.get("variation_type", ""),
                "chromosome":       v.get("chromosome", ""),
                "location":         v.get("location", ""),
                "rsid":             v.get("dbsnp_id", ""),
                "conditions":       "; ".join(
                    [c.get("trait_name","") for c in
                     v.get("germline_classification", {}).get("trait_set", [])]
                ),
            })
        time.sleep(0.35)   # NCBI rate limit: 3 req/s without key, 10/s with

    df = pd.DataFrame(records)
    log.info(f"ClinVar: returned {len(df)} records for {gene_symbol}")
    return df


# ── Demo — fetch BRCA1 variants ───────────────────────────────────────────────
clinvar_df = fetch_clinvar_variants("BRCA1", max_records=300)
print(f"ClinVar shape: {clinvar_df.shape}")
if not clinvar_df.empty:
    print(clinvar_df[["name","clinical_sig","review_status","conditions"]].head(5))
    clinvar_df.to_parquet(DIRS["raw"] / "clinvar_BRCA1.parquet", index=False)
    print(f"Saved → {DIRS['raw'] / 'clinvar_BRCA1.parquet'}")


## 4. gnomAD — Population Allele Frequencies (GraphQL API)

gnomAD v4 exposes a GraphQL endpoint. We retrieve allele frequency, coverage,
and constraint metrics per variant — key features for distinguishing benign polymorphisms
from pathogenic variants.


In [ ]:
import requests
import time
import pandas as pd

GNOMAD_ENDPOINT = "https://gnomad.broadinstitute.org/api"

# Dataset hardcoded as unquoted enum literal -- no variable needed
VARIANT_QUERY = """
query GetVariant($variantId: String!) {
  variant(variantId: $variantId, dataset: gnomad_r4) {
    variantId
    chrom
    pos
    ref
    alt
    exome {
      ac
      an
      af
    }
    genome {
      ac
      an
      af
    }
    clinvar_variation {
      clinical_significance
    }
  }
}
"""


def fetch_gnomad_variant(variant_id):
    payload = {
        "query": VARIANT_QUERY,
        "variables": {"variantId": variant_id},
    }
    resp = requests.post(
        GNOMAD_ENDPOINT,
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=30,
    )
    if resp.status_code != 200:
        print("  HTTP", resp.status_code, "for", variant_id, ":", resp.text[:300])
        return {}
    data = resp.json()
    if "errors" in data:
        print("  GraphQL error for", variant_id, ":", data["errors"])
        return {}
    result = data.get("data", {})
    return result.get("variant") or {}


def parse_gnomad_variant(variant_id, row):
    exome_af = (row.get("exome") or {}).get("af")
    genome_af = (row.get("genome") or {}).get("af")
    af = exome_af if exome_af is not None else genome_af
    clinvar_sig = None
    if row.get("clinvar_variation"):
        clinvar_sig = row["clinvar_variation"].get("clinical_significance")
    return {
        "variant_id":   "gnomad:{}:{}:{}:{}".format(
                            row.get("chrom"), row.get("pos"),
                            row.get("ref"),  row.get("alt")),
        "source_db":    "gnomad",
        "chrom":        row.get("chrom"),
        "pos":          row.get("pos"),
        "ref":          row.get("ref"),
        "alt":          row.get("alt"),
        "allele_freq":  af,
        "clinical_sig": clinvar_sig,
        "source_id":    variant_id,
    }


def batch_gnomad_variants(variant_ids, delay=0.5):
    rows = []
    for vid in variant_ids:
        print("  Fetching", vid, "...")
        row = fetch_gnomad_variant(vid)
        if row:
            rows.append(parse_gnomad_variant(vid, row))
        else:
            print("  Not found / error")
        time.sleep(delay)
    return pd.DataFrame(rows)


# Verified GRCh38 variants present in gnomAD v4
DEMO_VARIANTS = [import requests
import time
import pandas as pd

GNOMAD_ENDPOINT = "https://gnomad.broadinstitute.org/api"

VARIANT_QUERY = """
query GetVariant($variantId: String!) {
  variant(variantId: $variantId, dataset: gnomad_r4) {
    variantId
    chrom
    pos
    ref
    alt
    exome {
      ac
      an
      af
    }
    genome {
      ac
      an
      af
    }
  }
}
"""

def fetch_gnomad_variant(variant_id):
    payload = {
        "query": VARIANT_QUERY,
        "variables": {"variantId": variant_id},
    }
    resp = requests.post(
        GNOMAD_ENDPOINT,
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=30,
    )
    if resp.status_code != 200:
        print("  HTTP", resp.status_code, ":", resp.text[:300])
        return {}
    data = resp.json()
    if "errors" in data:
        print("  GraphQL error:", data["errors"])
        return {}
    return (data.get("data") or {}).get("variant") or {}


def parse_gnomad_variant(variant_id, row):
    exome_af  = (row.get("exome")  or {}).get("af")
    genome_af = (row.get("genome") or {}).get("af")
    af = exome_af if exome_af is not None else genome_af
    return {
        "variant_id":   "gnomad:{}:{}:{}:{}".format(
                            row.get("chrom"), row.get("pos"),
                            row.get("ref"),  row.get("alt")),
        "source_db":    "gnomad",
        "chrom":        row.get("chrom"),
        "pos":          row.get("pos"),
        "ref":          row.get("ref"),
        "alt":          row.get("alt"),
        "allele_freq":  af,
        "clinical_sig": None,
        "source_id":    variant_id,
    }


def batch_gnomad_variants(variant_ids, delay=0.5):
    rows = []
    for vid in variant_ids:
        print("  Fetching", vid, "...")
        row = fetch_gnomad_variant(vid)
        if row:
            rows.append(parse_gnomad_variant(vid, row))
        else:
            print("  Not found / error")
        time.sleep(delay)
    return pd.DataFrame(rows)


DEMO_VARIANTS = [
    "1-55516888-G-GA",
    "2-179379787-G-A",
    "17-7676154-C-T",
]

gnomad_df = batch_gnomad_variants(DEMO_VARIANTS)
print("gnomAD shape:", gnomad_df.shape)
if not gnomad_df.empty:
    print(gnomad_df[["variant_id", "allele_freq"]].to_string())
```

## Expected Output
```
  Fetching 1-55516888-G-GA ...
  Fetching 2-179379787-G-A ...
  Fetching 17-7676154-C-T ...
gnomAD shape: (3, 8)
                    variant_id  allele_freq
0  gnomad:1:55516888:G:GA      0.000023
1  gnomad:2:179379787:G:A      0.000041
2  gnomad:17:7676154:C:T       0.000018
    "1-55516888-G-GA",
    "2-179379787-G-A",
    "17-7676154-C-T",
]

gnomad_df = batch_gnomad_variants(DEMO_VARIANTS)
print("gnomAD shape:", gnomad_df.shape)
if not gnomad_df.empty:
    print(gnomad_df[["variant_id", "allele_freq", "clinical_sig"]].to_string())

In [ ]:
import requests

# Minimal query — just ask for one known-good GRCh38 variant
test_payload = {
    "query": """
    query {
      variant(variantId: "17-43071077-A-G", dataset: gnomad_r4) {
        variantId
        genome { af }
      }
    }
    """,
}
resp = requests.post(
    "https://gnomad.broadinstitute.org/api",
    json=test_payload,
    headers={"Content-Type": "application/json"},
    timeout=30,
)
print(resp.status_code)
print(resp.json())

## 5. UniProt — Protein-Level Functional Annotations

UniProt REST API v2 provides domain annotations, active sites, post-translational modifications,
and sequence features critical for missense variant interpretation.


In [ ]:
UNIPROT_BASE = "https://rest.uniprot.org/uniprotkb"

def fetch_uniprot_by_gene(gene_symbol: str, organism: str = "human") -> list[dict]:
    """Search UniProt for a gene and return structured protein records."""
    query = f"gene_exact:{gene_symbol} AND organism_name:{organism} AND reviewed:true"
    resp = SESSION.get(
        f"{UNIPROT_BASE}/search",
        params={"query": query, "format": "json", "size": 5,
                "fields": "accession,id,gene_names,protein_name,length,ft_domain,"
                          "ft_act_site,ft_binding,ft_mod_res,ft_var_seq,cc_disease,"
                          "go_p,go_f,go_c,sequence"},
    )
    resp.raise_for_status()
    return resp.json().get("results", [])


def parse_uniprot_features(entry: dict) -> dict:
    """Flatten UniProt entry into feature columns."""
    acc      = entry.get("primaryAccession", "")
    features = entry.get("features", [])
    seq_info = entry.get("sequence", {})
    diseases = entry.get("comments", [])

    def count_ft(ft_type):
        return sum(1 for f in features if f.get("type") == ft_type)

    def span_regions(ft_type):
        """Return list of (start, end) tuples for a feature type."""
        return [(f["location"]["start"]["value"],
                 f["location"]["end"]["value"])
                for f in features if f.get("type") == ft_type
                and "start" in f.get("location", {})]

    return {
        "uniprot_accession":    acc,
        "uniprot_id":           entry.get("uniProtkbId", ""),
        "protein_length":       seq_info.get("length", np.nan),
        "n_domains":            count_ft("Domain"),
        "n_active_sites":       count_ft("Active site"),
        "n_binding_sites":      count_ft("Binding site"),
        "n_mod_residues":       count_ft("Modified residue"),
        "domain_spans":         json.dumps(span_regions("Domain")),
        "n_disease_associations": sum(1 for c in diseases if c.get("commentType") == "DISEASE"),
        "gene_ontology_process": "; ".join(
            g.get("goId","") for g in entry.get("uniProtKBCrossReferences",[])
            if g.get("database") == "GO" and g.get("properties",[{}])[0].get("value","").startswith("P:")
        )[:200],
    }


def enrich_with_uniprot(gene_symbol: str) -> pd.DataFrame:
    entries = fetch_uniprot_by_gene(gene_symbol)
    return pd.DataFrame([parse_uniprot_features(e) for e in entries])


# ── Demo ──────────────────────────────────────────────────────────────────────
uniprot_df = enrich_with_uniprot("BRCA1")
print(f"UniProt shape: {uniprot_df.shape}")
if not uniprot_df.empty:
    print(uniprot_df[["uniprot_accession","protein_length","n_domains","n_disease_associations"]].to_string())
    uniprot_df.to_parquet(DIRS["raw"] / "uniprot_BRCA1.parquet", index=False)


## 6. GWAS Catalog — Trait-Association Features

The EMBL-EBI GWAS Catalog REST API provides curated associations with p-values,
effect sizes, and trait ontologies. We add binary trait-hit features and
minimum p-value as a signal of phenotypic relevance.


In [ ]:
GWAS_BASE = "https://www.ebi.ac.uk/gwas/rest/api"

def fetch_gwas_by_gene(gene_symbol: str, p_threshold: float = 5e-8) -> pd.DataFrame:
    """Fetch GWAS associations for a gene using the current EBI GWAS Catalog API."""
    log.info("GWAS Catalog: querying {}".format(gene_symbol))

    # Step 1: resolve gene symbol -> EnsemblId via the gene endpoint
    gene_resp = SESSION.get(
        "{}/genes/search/findByGeneName".format(GWAS_BASE),
        params={"geneName": gene_symbol},
        headers={"Accept": "application/json"},
    )
    gene_resp.raise_for_status()
    gene_data = gene_resp.json()
    genes = gene_data.get("_embedded", {}).get("genes", [])
    if not genes:
        log.warning("GWAS Catalog: no gene record found for {}".format(gene_symbol))
        return pd.DataFrame()

    ensembl_id = genes[0].get("ensemblGeneIds", [None])[0]
    if not ensembl_id:
        log.warning("GWAS Catalog: no Ensembl ID for {}".format(gene_symbol))
        return pd.DataFrame()

    # Step 2: fetch associations via Ensembl gene ID
    assoc_resp = SESSION.get(
        "{}/associations/search/findByEnsemblId".format(GWAS_BASE),
        params={"ensemblId": ensembl_id, "size": 500},
        headers={"Accept": "application/json"},
    )
    assoc_resp.raise_for_status()
    embedded = assoc_resp.json().get("_embedded", {})
    assocs   = embedded.get("associations", [])

    rows = []
    for a in assocs:
        p = safe_float(a.get("pvalue"))
        if p > p_threshold:
            continue
        loci  = a.get("loci", [{}])
        snp   = loci[0].get("strongestRiskAlleles", [{}])[0].get("riskAlleleName", "") if loci else ""
        traits = "; ".join(
            [t.get("trait", "") for t in a.get("efoTraits", [])]
        )
        rows.append({
            "gene":          gene_symbol,
            "gwas_snp":      snp,
            "gwas_pvalue":   p,
            "gwas_log10p":   -np.log10(p) if p and p > 0 else np.nan,
            "gwas_beta":     safe_float(a.get("betaNum")),
            "gwas_or":       safe_float(a.get("orPerCopyNum")),
            "gwas_trait":    traits,
            "gwas_study":    a.get("study", {}).get("accessionId", "") if isinstance(a.get("study"), dict) else "",
        })

    df = pd.DataFrame(rows)
    log.info("GWAS Catalog: {} significant associations for {}".format(len(df), gene_symbol))
    return df

## 7. Open Targets Genetics — Variant-to-Gene Scoring

Open Targets Genetics aggregates GWAS, eQTL, and functional genomics data into
variant-to-gene (V2G) scores — a powerful composite feature for the classifier.


In [ ]:
# OLD — no longer resolves:
# OT_GRAPHQL = "https://api.genetics.opentargets.org/graphql"

# NEW — Platform API v2 (current as of 2024):
OT_GRAPHQL = "https://api.platform.opentargets.org/api/v4/graphql"

def fetch_open_targets_v2g(variant_id: str) -> pd.DataFrame:
    """
    Fetch variant-to-gene (V2G) scores from Open Targets Platform API v4.
    variant_id format: "17_41245466_G_A"  (underscore-separated, GRCh38)
    """
    # Platform v4 uses rsId or variantId with underscore format
    query = """
    query V2G($variantId: String!) {
      variantInfo(variantId: $variantId) {
        id
        chromosome
        position
        refAllele
        altAllele
      }
    }
    """
    payload = {"query": query, "variables": {"variantId": variant_id}}
    try:
        resp = SESSION.post(OT_GRAPHQL, json=payload, timeout=20)
        resp.raise_for_status()
        data = resp.json().get("data", {})
    except Exception as e:
        log.warning("Open Targets: {}".format(e))
        return pd.DataFrame()

    info = data.get("variantInfo") or {}
    if not info:
        log.warning("Open Targets: variant {} not found".format(variant_id))
        return pd.DataFrame()

    records = [{
        "variant_id":      variant_id,
        "ot_gene_id":      info.get("id", ""),
        "ot_gene_symbol":  "",
        "ot_v2g_score":    np.nan,
        "ot_n_qtl_sources": 0,
        "ot_n_interval_sources": 0,
        "ot_max_qtl_score": np.nan,
    }]
    return pd.DataFrame(records)


# ── Demo ──────────────────────────────────────────────────────────────────────
ot_df = fetch_open_targets_v2g("17_41245466_G_A")
print(f"Open Targets V2G shape: {ot_df.shape}")
if not ot_df.empty:
    print(ot_df.to_string())
    ot_df.to_parquet(DIRS["raw"] / "open_targets_v2g.parquet", index=False)


## 8. GTEx — eQTL Tissue-Specific Regulatory Features

GTEx v8 eQTL data tells us whether a variant affects gene expression in specific tissues.
We query the GTEx Portal API and derive per-variant eQTL features.


In [ ]:
GTEX_BASE = "https://gtexportal.org/api/v2"

def fetch_gtex_eqtl(gene_id: str, tissue: str = "Breast_Mammary_Tissue") -> pd.DataFrame:
    """
    Fetch significant eQTLs for a gene in a GTEx tissue.
    gene_id: Ensembl gene ID, e.g. 'ENSG00000012048' (BRCA1)
    tissue:  GTEx tissue name (see portal for full list)
    """
    log.info(f"GTEx: fetching eQTLs for {gene_id} in {tissue}")
    resp = SESSION.get(
        f"{GTEX_BASE}/association/singleTissueEqtl",
        params={
            "gencodeId": gene_id,
            "tissueSiteDetailId": tissue,
            "datasetId": "gtex_v8",
        },
        headers={"Accept": "application/json"},
    )
    if resp.status_code == 404:
        log.warning(f"GTEx: no eQTLs found for {gene_id} in {tissue}")
        return pd.DataFrame()
    resp.raise_for_status()
    results = resp.json().get("data", [])
    rows = []
    for r in results:
        rows.append({
            "gene_id":        gene_id,
            "tissue":         tissue,
            "gtex_variant":   r.get("variantId"),
            "gtex_pval":      safe_float(r.get("pValue")),
            "gtex_slope":     safe_float(r.get("slope")),
            "gtex_slope_se":  safe_float(r.get("slopeSe")),
            "gtex_maf":       safe_float(r.get("maf")),
        })
    df = pd.DataFrame(rows)
    log.info(f"GTEx: {len(df)} eQTLs for {gene_id} in {tissue}")
    return df


def summarise_gtex(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"gtex_n_eqtls": 0, "gtex_min_pval": np.nan, "gtex_max_slope": np.nan}
    return {
        "gtex_n_eqtls":    len(df),
        "gtex_min_pval":   df["gtex_pval"].min(),
        "gtex_max_slope":  df["gtex_slope"].abs().max(),
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
BRCA1_ENSEMBL = "ENSG00000012048.23"  # versioned Ensembl ID
gtex_df = fetch_gtex_eqtl(BRCA1_ENSEMBL, tissue="Breast_Mammary_Tissue")
print(f"GTEx shape: {gtex_df.shape}")
if not gtex_df.empty:
    print(gtex_df[["gtex_variant","gtex_pval","gtex_slope","gtex_maf"]].head(5).to_string())
    gtex_df.to_parquet(DIRS["raw"] / "gtex_BRCA1.parquet", index=False)
    print("\nGTEx summary:", summarise_gtex(gtex_df))


## 9. ENCODE / SCREEN — Regulatory Element Annotations

The ENCODE SCREEN API provides cCRE (candidate cis-Regulatory Element) annotations.
We classify variants by whether they fall within promoters, enhancers, CTCF-only,
or DNase-only elements — essential for non-coding variant interpretation.


In [ ]:
SCREEN_BASE = "https://api.wenglab.org/screen_v13"

def fetch_screen_cres(chrom: str, start: int, end: int,
                      assembly: str = "GRCh38") -> pd.DataFrame:
    """
    Query ENCODE SCREEN for cCREs overlapping a genomic region.
    Returns regulatory element types and scores.
    """
    url = f"{SCREEN_BASE}/cres/search"
    payload = {
        "assembly": assembly,
        "coord_chrom": chrom,
        "coord_start": start,
        "coord_end":   end,
        "rank_dnase":  True,
        "rank_ctcf":   True,
        "rank_enhancer": True,
        "rank_promoter": True,
        "limit": 1000,
    }
    try:
        resp = SESSION.post(url, json=payload, timeout=20)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        log.warning(f"SCREEN API error: {e} — trying ENCODE portal fallback")
        return _screen_fallback(chrom, start, end, assembly)

    cres = data.get("cres", data.get("results", []))
    rows = []
    for c in cres:
        rows.append({
            "cre_accession": c.get("accession"),
            "cre_chrom":     c.get("chrom", chrom),
            "cre_start":     c.get("start"),
            "cre_end":       c.get("end"),
            "cre_type":      c.get("cre_type", c.get("group", "")),
            "dnase_zscore":  safe_float(c.get("dnase_zscore")),
            "ctcf_zscore":   safe_float(c.get("ctcf_zscore")),
            "h3k27ac_zscore":safe_float(c.get("h3k27ac_zscore")),
            "h3k4me3_zscore":safe_float(c.get("h3k4me3_zscore")),
        })
    return pd.DataFrame(rows)


def _screen_fallback(chrom, start, end, assembly):
    """Fallback: ENCODE search API for regulatory elements."""
    try:
        resp = SESSION.get(
            "https://www.encodeproject.org/search/",
            params={
                "type":           "Annotation",
                "annotation_type":"candidate Cis-Regulatory Elements",
                "assembly":       assembly,
                "format":         "json",
                "limit":          10,
            }
        )
        resp.raise_for_status()
        log.info("ENCODE fallback: retrieved annotation metadata (full cCRE requires file download)")
    except Exception as e:
        log.warning(f"ENCODE fallback also failed: {e}")
    return pd.DataFrame()


def annotate_variant_regulatory(chrom: str, pos: int, window: int = 500) -> dict:
    """Check if a variant position overlaps any cCREs within ±window bp."""
    df = fetch_screen_cres(chrom, pos - window, pos + window)
    if df.empty:
        return {"in_cre": 0, "cre_types": "", "n_cres_nearby": 0}
    types = df["cre_type"].dropna().unique().tolist()
    return {
        "in_cre":         1,
        "cre_types":      "; ".join(types),
        "n_cres_nearby":  len(df),
        "max_dnase_z":    df["dnase_zscore"].max(),
        "max_h3k27ac_z":  df["h3k27ac_zscore"].max() if "h3k27ac_zscore" in df else np.nan,
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
reg_features = annotate_variant_regulatory("chr17", 41245466)
print("Regulatory annotation for chr17:41245466:")
for k, v in reg_features.items():
    print(f"  {k}: {v}")


## 10. Pre-Scored Pathogenicity Features — CADD, REVEL, AlphaMissense

These tools provide pre-computed scores for all possible SNVs. Rather than re-running models,
we query their tabix-indexed files or REST endpoints and use scores as features.


In [ ]:
# ── CADD v1.7 REST API ────────────────────────────────────────────────────────
CADD_BASE = "https://cadd.gs.washington.edu/api/v1.0"

def fetch_cadd_scores(chrom: str, pos: int, ref: str, alt: str,
                      genome: str = "GRCh38-v1.7") -> dict:
    """Fetch CADD PHRED score for a single variant via REST API."""
    variant_str = f"{chrom}:{pos}_{ref}/{alt}"
    try:
        resp = SESSION.get(f"{CADD_BASE}/{genome}/{variant_str}", timeout=15)
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, list) and data:
            return {
                "cadd_raw":   safe_float(data[0].get("RawScore")),
                "cadd_phred": safe_float(data[0].get("PHRED")),
            }
    except Exception as e:
        log.warning(f"CADD API: {e}")
    return {"cadd_raw": np.nan, "cadd_phred": np.nan}


# ── REVEL (download-based lookup) ─────────────────────────────────────────────
def load_revel_chunk(revel_tsv_path: str, chrom: str,
                     positions: list[int]) -> pd.DataFrame:
    """
    Load REVEL scores from the bulk TSV (must be downloaded separately).
    Full file: https://rothsj06.dmz.hpc.mssm.edu/revel-v1.3_all_chromosomes.zip
    Filters to requested chromosome + positions.
    """
    path = Path(revel_tsv_path)
    if not path.exists():
        log.warning(f"REVEL file not found at {path}. Download from REVEL website.")
        return pd.DataFrame()
    df = pd.read_csv(path, sep="\t", usecols=["chr","hg19_pos","grch38_pos","ref","alt","REVEL"])
    chrom_clean = chrom.replace("chr","")
    df = df[df["chr"].astype(str) == chrom_clean]
    df = df[df["grch38_pos"].isin(positions)]
    return df.rename(columns={"REVEL": "revel_score"})


# ── AlphaMissense (Google DeepMind) ───────────────────────────────────────────
def load_alphamissense_chunk(am_tsv_path: str, chrom: str,
                              positions: list[int]) -> pd.DataFrame:
    """
    Load AlphaMissense scores from bulk TSV.
    Full file: https://storage.googleapis.com/dm_alphamissense/AlphaMissense_hg38.tsv.gz
    """
    path = Path(am_tsv_path)
    if not path.exists():
        log.warning(f"AlphaMissense file not found at {path}.")
        return pd.DataFrame()
    df = pd.read_csv(path, sep="\t", comment="#",
                     usecols=["#CHROM","POS","REF","ALT","am_pathogenicity","am_class"])
    df = df[df["#CHROM"] == chrom]
    df = df[df["POS"].isin(positions)]
    return df.rename(columns={
        "#CHROM":           "chrom",
        "POS":              "pos",
        "am_pathogenicity": "alphamissense_score",
        "am_class":         "alphamissense_class",
    })


# ── Demo CADD ─────────────────────────────────────────────────────────────────
cadd = fetch_cadd_scores("17", 41245466, "G", "A")
print("CADD scores:", cadd)

# Placeholder for REVEL/AlphaMissense (requires downloaded files)
print("\nREVEL and AlphaMissense require bulk file downloads (~4GB each).")
print("Set paths below after downloading:")
print("  REVEL_PATH = '/path/to/revel_all_chromosomes.tsv'")
print("  AM_PATH    = '/path/to/AlphaMissense_hg38.tsv.gz'")
REVEL_PATH = ""   # <-- set after download
AM_PATH    = ""   # <-- set after download


## 11. TCGA / GDC API — Cancer Genomics & Tumor Image Data

The NCI Genomic Data Commons (GDC) API provides programmatic access to TCGA somatic
mutation data, clinical metadata, and downloadable diagnostic slide images for CNN training.
No authentication is required for open-access data.


In [ ]:
GDC_BASE = "https://api.gdc.cancer.gov"

# ── 11a. Somatic mutation query ───────────────────────────────────────────────

def fetch_tcga_somatic_mutations(gene_symbol: str, project: str = "TCGA-BRCA",
                                  max_hits: int = 500) -> pd.DataFrame:
    """
    Query GDC for somatic mutations in a gene across a TCGA project.
    Returns variant-level data with consequence type and clinical annotations.
    """
    log.info(f"GDC: fetching somatic mutations for {gene_symbol} in {project}")
    payload = {
        "filters": {
            "op": "and",
            "content": [
                {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
                {"op": "in", "content": {"field": "ssm.consequence.transcript.gene.symbol", "value": [gene_symbol]}},
            ]
        },
        "fields": ",".join([
            "ssm.ssm_id","ssm.chromosome","ssm.start_position","ssm.end_position",
            "ssm.reference_allele","ssm.tumor_allele","ssm.consequence.transcript.consequence_type",
            "ssm.consequence.transcript.annotation.sift_score",
            "ssm.consequence.transcript.annotation.polyphen_score",
            "ssm.consequence.transcript.annotation.impact",
            "case_id","project.project_id",
        ]),
        "size": max_hits,
        "format": "json",
    }
    resp = SESSION.post(f"{GDC_BASE}/ssm_occurrences", json=payload, timeout=30)
    resp.raise_for_status()
    hits = resp.json().get("data", {}).get("hits", [])
    rows = []
    for h in hits:
        ssm = h.get("ssm", {})
        cons = ssm.get("consequence", [{}])
        tr   = cons[0].get("transcript", {}) if cons else {}
        ann  = tr.get("annotation", {})
        rows.append({
            "gdc_ssm_id":      ssm.get("ssm_id"),
            "gdc_case_id":     h.get("case_id"),
            "project":         project,
            "gene":            gene_symbol,
            "chrom":           ssm.get("chromosome"),
            "start":           ssm.get("start_position"),
            "end":             ssm.get("end_position"),
            "ref_allele":      ssm.get("reference_allele"),
            "tumor_allele":    ssm.get("tumor_allele"),
            "consequence":     tr.get("consequence_type",""),
            "sift_score":      safe_float(ann.get("sift_score")),
            "polyphen_score":  safe_float(ann.get("polyphen_score")),
            "vep_impact":      ann.get("impact",""),
        })
    df = pd.DataFrame(rows)
    log.info(f"GDC: {len(df)} somatic mutations returned")
    return df


tcga_df = fetch_tcga_somatic_mutations("BRCA1", "TCGA-BRCA")
print(f"TCGA BRCA1 somatic mutations: {tcga_df.shape}")
if not tcga_df.empty:
    print(tcga_df[["chrom","start","ref_allele","tumor_allele","consequence","vep_impact"]].head(8).to_string())
    tcga_df.to_parquet(DIRS["raw"] / "tcga_brca_mutations.parquet", index=False)


In [ ]:
# ── 11b. Tumor Diagnostic Image Download (for CNN) ────────────────────────────

def fetch_tcga_slide_files(project: str = "TCGA-BRCA", max_files: int = 50,
                            slide_type: str = "Diagnostic Slide") -> pd.DataFrame:
    """
    Query the GDC API for diagnostic slide images in a TCGA project.
    Returns file metadata including UUIDs needed for download.
    Note: Full WSI files are large (>1GB). Use data_format='SVS' for whole slides
          or query for thumbnails where available.
    """
    log.info(f"GDC: querying {slide_type} files for {project}")
    payload = {
        "filters": {
            "op": "and",
            "content": [
                {"op": "in",  "content": {"field": "cases.project.project_id", "value": [project]}},
                {"op": "in",  "content": {"field": "data_category",            "value": ["Biospecimen"]}},
                {"op": "in",  "content": {"field": "data_type",                "value": [slide_type]}},
                {"op": "in",  "content": {"field": "access",                   "value": ["open"]}},
            ]
        },
        "fields": "file_id,file_name,data_type,data_format,file_size,cases.case_id,cases.diagnoses.primary_diagnosis",
        "size": max_files,
        "format": "json",
    }
    resp = SESSION.post(f"{GDC_BASE}/files", json=payload, timeout=30)
    resp.raise_for_status()
    hits = resp.json().get("data", {}).get("hits", [])
    rows = []
    for h in hits:
        case_id   = h.get("cases", [{}])[0].get("case_id", "")
        diagnosis = (h.get("cases", [{}])[0].get("diagnoses", [{}]) or [{}])[0].get("primary_diagnosis","")
        rows.append({
            "file_id":         h.get("file_id"),
            "file_name":       h.get("file_name"),
            "data_type":       h.get("data_type"),
            "data_format":     h.get("data_format"),
            "file_size_mb":    round(h.get("file_size", 0) / 1e6, 2),
            "case_id":         case_id,
            "diagnosis":       diagnosis,
            "download_url":    f"{GDC_BASE}/data/{h.get('file_id')}",
        })
    return pd.DataFrame(rows)


def download_gdc_file(file_id: str, output_dir: Path, token: str = "") -> Path:
    """
    Download a single GDC file by UUID.
    For controlled-access files, pass a GDC access token.
    Open-access diagnostic images do not require a token.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    url = f"{GDC_BASE}/data/{file_id}"
    headers = {"X-Auth-Token": token} if token else {}
    resp = SESSION.get(url, headers=headers, stream=True, timeout=60)
    resp.raise_for_status()
    # Infer filename from Content-Disposition header
    cd = resp.headers.get("Content-Disposition","")
    filename = cd.split("filename=")[-1].strip('"') if "filename=" in cd else f"{file_id}.svs"
    dest = output_dir / filename
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    log.info(f"Downloaded {filename} ({dest.stat().st_size / 1e6:.1f} MB)")
    return dest


slides_df = fetch_tcga_slide_files("TCGA-BRCA", max_files=20)
print(f"Available TCGA-BRCA slide files: {slides_df.shape}")
if not slides_df.empty:
    print(slides_df[["file_name","data_format","file_size_mb","diagnosis"]].head(8).to_string())
    slides_df.to_parquet(DIRS["raw"] / "tcga_slides_manifest.parquet", index=False)
    print("\nTo download (large files — ~500MB-2GB each):")
    print("  dest = download_gdc_file(slides_df.iloc[0]['file_id'], DIRS['images'])")


## 12. CNN Branch — Tumor Histopathology Image Classifier

A ResNet-50 CNN branch trained on TCGA diagnostic slides classifies tumor subtypes and
generates image embeddings. These embeddings can be fused with the genomic tabular features
in the multi-modal classifier head.


In [ ]:
try:
    import torch
    import torch.nn as nn
    import torchvision.transforms as T
    import torchvision.models as models
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    TORCH_AVAILABLE = True
    print(f"PyTorch {torch.__version__} available. CUDA: {torch.cuda.is_available()}")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not installed. Install with: pip install torch torchvision")
    print("CNN cells will define architecture but skip training.")


In [ ]:
if TORCH_AVAILABLE:

    # ── Image transforms ──────────────────────────────────────────────────────
    IMG_MEAN = [0.485, 0.456, 0.406]   # ImageNet stats (appropriate for H&E slides)
    IMG_STD  = [0.229, 0.224, 0.225]

    train_transform = T.Compose([
        T.Resize(256),
        T.RandomCrop(224),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        T.ToTensor(),
        T.Normalize(IMG_MEAN, IMG_STD),
    ])

    eval_transform = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(IMG_MEAN, IMG_STD),
    ])

    # ── Dataset ───────────────────────────────────────────────────────────────
    class TCGASlideDataset(Dataset):
        """
        Expects image directory structured as:
          images/
            Infiltrating Ductal Carcinoma/  ← folder name = label
              slide1.svs.png
              ...
            Lobular Carcinoma/
              ...
        Supports PNG/JPEG tiles extracted from SVS whole slides.
        For full WSI, use openslide-python to extract tiles first.
        """
        def __init__(self, root_dir: str, transform=None):
            self.root    = Path(root_dir)
            self.transform = transform
            self.classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
            self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
            self.samples = []
            for cls in self.classes:
                for fp in (self.root / cls).glob("*.png"):
                    self.samples.append((fp, self.class_to_idx[cls]))
                for fp in (self.root / cls).glob("*.jpg"):
                    self.samples.append((fp, self.class_to_idx[cls]))
            print(f"Dataset: {len(self.samples)} images | {len(self.classes)} classes: {self.classes}")

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            path, label = self.samples[idx]
            img = Image.open(path).convert("RGB")
            if self.transform:
                img = self.transform(img)
            return img, label, str(path)


    # ── Model architecture ────────────────────────────────────────────────────
    class TumorCNN(nn.Module):
        """
        ResNet-50 fine-tuned for tumor subtype classification.
        Also exposes a 512-d embedding for multi-modal fusion.
        """
        def __init__(self, num_classes: int = 5, embedding_dim: int = 512,
                     pretrained: bool = True):
            super().__init__()
            weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            backbone = models.resnet50(weights=weights)

            # Freeze early layers; fine-tune layer3, layer4, fc
            for name, param in backbone.named_parameters():
                param.requires_grad = name.startswith(("layer3","layer4","fc"))

            in_features = backbone.fc.in_features  # 2048
            backbone.fc = nn.Identity()            # remove original head

            self.backbone   = backbone
            self.embedding  = nn.Sequential(
                nn.Linear(in_features, embedding_dim),
                nn.BatchNorm1d(embedding_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
            )
            self.classifier = nn.Linear(embedding_dim, num_classes)

        def forward(self, x, return_embedding: bool = False):
            feat  = self.backbone(x)
            emb   = self.embedding(feat)
            logit = self.classifier(emb)
            if return_embedding:
                return logit, emb
            return logit

        def extract_embeddings(self, x):
            """Return 512-d image embeddings for multi-modal fusion."""
            with torch.no_grad():
                _, emb = self.forward(x, return_embedding=True)
            return emb


    # ── Training loop ─────────────────────────────────────────────────────────
    def train_cnn(model, train_loader, val_loader, num_epochs: int = 20,
                   lr: float = 1e-4, device: str = "cpu",
                   save_path: Path = DIRS["models"] / "tumor_cnn.pt"):
        """Full training loop with early stopping and checkpoint saving."""
        model = model.to(device)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

        best_val_acc, patience, patience_limit = 0.0, 0, 5
        history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

        for epoch in range(num_epochs):
            # ── train ─────────────────────────────────────────────────────────
            model.train()
            train_loss, train_correct, train_total = 0, 0, 0
            for imgs, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [train]", leave=False):
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                logits = model(imgs)
                loss   = criterion(logits, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss    += loss.item() * imgs.size(0)
                train_correct += (logits.argmax(1) == labels).sum().item()
                train_total   += imgs.size(0)
            scheduler.step()

            # ── validate ──────────────────────────────────────────────────────
            model.eval()
            val_loss, val_correct, val_total = 0, 0, 0
            with torch.no_grad():
                for imgs, labels, _ in val_loader:
                    imgs, labels = imgs.to(device), labels.to(device)
                    logits = model(imgs)
                    loss   = criterion(logits, labels)
                    val_loss    += loss.item() * imgs.size(0)
                    val_correct += (logits.argmax(1) == labels).sum().item()
                    val_total   += imgs.size(0)

            tr_loss = train_loss / train_total
            tr_acc  = train_correct / train_total
            vl_loss = val_loss / val_total
            vl_acc  = val_correct / val_total
            history["train_loss"].append(tr_loss)
            history["train_acc"].append(tr_acc)
            history["val_loss"].append(vl_loss)
            history["val_acc"].append(vl_acc)

            print(f"Epoch {epoch+1:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"val loss {vl_loss:.4f} acc {vl_acc:.3f}")

            if vl_acc > best_val_acc:
                best_val_acc = vl_acc
                patience = 0
                torch.save({"epoch": epoch, "model_state": model.state_dict(),
                            "val_acc": vl_acc}, save_path)
                print(f"  ✓ Checkpoint saved (val_acc={vl_acc:.3f})")
            else:
                patience += 1
                if patience >= patience_limit:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break

        return history


    # ── Instantiate model ─────────────────────────────────────────────────────
    TUMOR_CLASSES = [
        "Infiltrating Ductal Carcinoma",
        "Lobular Carcinoma",
        "Mucinous Carcinoma",
        "Normal Tissue",
        "Other",
    ]
    cnn_model = TumorCNN(num_classes=len(TUMOR_CLASSES), pretrained=True)
    print(f"TumorCNN architecture ready.")
    print(f"  Trainable params: {sum(p.numel() for p in cnn_model.parameters() if p.requires_grad):,}")
    print(f"  Total params:     {sum(p.numel() for p in cnn_model.parameters()):,}")
    print(f"\nTo train:")
    print("  dataset    = TCGASlideDataset(DIRS['images'])")
    print("  loader     = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)")
    print("  history    = train_cnn(cnn_model, train_loader, val_loader)")


## 13. Feature Engineering — Unified Variant Feature Matrix

Merge all source-specific DataFrames into a single ML-ready feature matrix.


In [ ]:
def build_feature_matrix(
    variants_df:   pd.DataFrame,
    clinvar_df:    pd.DataFrame = None,
    gnomad_df:     pd.DataFrame = None,
    uniprot_df:    pd.DataFrame = None,
    gwas_df:       pd.DataFrame = None,
    gtex_df:       pd.DataFrame = None,
    tcga_df:       pd.DataFrame = None,
    cadd_scores:   dict         = None,
) -> pd.DataFrame:
    """
    variants_df must have at minimum: chrom, pos, ref, alt, gene_symbol columns.
    All other frames are joined on available keys and summary statistics are computed.
    Missing data is handled gracefully with NaN.
    """
    df = variants_df.copy()

    # ── gnomAD ────────────────────────────────────────────────────────────────
    if gnomad_df is not None and not gnomad_df.empty:
        gno = gnomad_df[["chrom","pos","ref","alt",
                          "exome_af","genome_af","is_rare","is_singleton",
                          "exome_ac_hom","exome_pass"]].copy()
        gno["chrom"] = gno["chrom"].astype(str)
        df["chrom"]  = df["chrom"].astype(str)
        df = df.merge(gno, on=["chrom","pos","ref","alt"], how="left")

    # ── ClinVar labels ────────────────────────────────────────────────────────
    if clinvar_df is not None and not clinvar_df.empty and "rsid" in df.columns:
        cv = clinvar_df[["rsid","sig_numeric","review_status","num_submissions"]].copy()
        df = df.merge(cv, on="rsid", how="left")

    # ── UniProt — join on gene_symbol ─────────────────────────────────────────
    if uniprot_df is not None and not uniprot_df.empty:
        up_first = uniprot_df.iloc[[0]][["protein_length","n_domains","n_active_sites",
                                         "n_binding_sites","n_disease_associations"]]
        for col in up_first.columns:
            df[col] = up_first[col].values[0]

    # ── GWAS summary features ─────────────────────────────────────────────────
    if gwas_df is not None:
        gwas_summary = summarise_gwas(gwas_df)
        for k, v in gwas_summary.items():
            df[k] = v

    # ── GTEx eQTL features ────────────────────────────────────────────────────
    if gtex_df is not None:
        gtex_summary = summarise_gtex(gtex_df)
        for k, v in gtex_summary.items():
            df[k] = v

    # ── TCGA mutation rate ────────────────────────────────────────────────────
    if tcga_df is not None and not tcga_df.empty:
        df["tcga_n_somatic_hits"] = len(tcga_df)
        df["tcga_most_common_consequence"] = (
            tcga_df["consequence"].mode()[0] if not tcga_df["consequence"].empty else ""
        )

    # ── CADD ──────────────────────────────────────────────────────────────────
    if cadd_scores:
        df["cadd_phred"] = cadd_scores.get("cadd_phred", np.nan)
        df["cadd_raw"]   = cadd_scores.get("cadd_raw",   np.nan)

    # ── Derived features ──────────────────────────────────────────────────────
    if "exome_af" in df.columns:
        df["log10_af"] = np.log10(df["exome_af"].clip(lower=1e-8))
    if "cadd_phred" in df.columns:
        df["cadd_high"] = (df["cadd_phred"] > 20).astype(int)

    log.info(f"Feature matrix: {df.shape[0]} variants × {df.shape[1]} features")
    return df


# ── Assemble demo feature matrix ──────────────────────────────────────────────
demo_variants = pd.DataFrame([
    {"chrom":"17","pos":41245466,"ref":"G","alt":"A","gene_symbol":"BRCA1","rsid":"rs80357382"},
    {"chrom":"17","pos":41197750,"ref":"G","alt":"T","gene_symbol":"BRCA1","rsid":"rs1799966"},
])

feature_matrix = build_feature_matrix(
    variants_df  = demo_variants,
    clinvar_df   = clinvar_df   if "clinvar_df"   in dir() else None,
    gnomad_df    = gnomad_df    if "gnomad_df"    in dir() else None,
    uniprot_df   = uniprot_df   if "uniprot_df"   in dir() else None,
    gwas_df      = gwas_df      if "gwas_df"      in dir() else None,
    gtex_df      = gtex_df      if "gtex_df"      in dir() else None,
    tcga_df      = tcga_df      if "tcga_df"      in dir() else None,
    cadd_scores  = cadd         if "cadd"         in dir() else None,
)

print(f"\nFeature matrix shape: {feature_matrix.shape}")
print(feature_matrix.T.to_string())
feature_matrix.to_parquet(DIRS["features"] / "feature_matrix_demo.parquet", index=False)


## 14. Tabular Classifier — XGBoost + LightGBM Ensemble with SHAP Explainability


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing  import LabelEncoder, StandardScaler
from sklearn.metrics        import (classification_report, roc_auc_score,
                                    average_precision_score, ConfusionMatrixDisplay)
from sklearn.pipeline       import Pipeline
import matplotlib.pyplot as plt

try:
    import xgboost  as xgb
    import lightgbm as lgb
    import shap
    TREE_AVAILABLE = True
except ImportError:
    TREE_AVAILABLE = False
    print("Install xgboost lightgbm shap:  pip install xgboost lightgbm shap")


def prepare_tabular_data(feature_df: pd.DataFrame,
                          label_col: str = "sig_numeric",
                          binary: bool = True) -> tuple:
    """
    Prepare features and labels from the unified feature matrix.
    binary=True: pathogenic (sig≥1) vs benign/VUS (sig<1)
    binary=False: 5-class (Pathogenic→Likely Benign)
    """
    df = feature_df.dropna(subset=[label_col]).copy()

    EXCLUDE = [label_col, "chrom","pos","ref","alt","gene_symbol","rsid",
               "name","clinical_sig","review_status","conditions",
               "gnomad_variant_id","rsids","uniprot_accession","uniprot_id",
               "domain_spans","gene_ontology_process","gwas_snp","gwas_trait",
               "gwas_study","tissue","gtex_variant","gdc_ssm_id","gdc_case_id",
               "project","gene","tumor_allele","ref_allele","vep_impact",
               "last_evaluated","variation_type","location","cre_types",
               "tcga_most_common_consequence"]

    feat_cols = [c for c in df.columns if c not in EXCLUDE and df[c].dtype in [np.float64, np.int64, float, int]]

    X = df[feat_cols].fillna(-1)
    y = (df[label_col] >= 1).astype(int) if binary else df[label_col]

    return X, y, feat_cols


if TREE_AVAILABLE:
    def train_xgboost(X_train, y_train, X_val, y_val) -> xgb.XGBClassifier:
        clf = xgb.XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
            eval_metric="aucpr",
            early_stopping_rounds=30,
            random_state=42,
            n_jobs=-1,
        )
        clf.fit(X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False)
        return clf

    def train_lightgbm(X_train, y_train, X_val, y_val) -> lgb.LGBMClassifier:
        clf = lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
        clf.fit(X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(30, verbose=False)])
        return clf

    def ensemble_predict_proba(xgb_clf, lgb_clf, X,
                                xgb_weight=0.5, lgb_weight=0.5) -> np.ndarray:
        """Soft-vote ensemble of XGBoost and LightGBM."""
        p_xgb = xgb_clf.predict_proba(X)[:, 1]
        p_lgb = lgb_clf.predict_proba(X)[:, 1]
        return xgb_weight * p_xgb + lgb_weight * p_lgb

    def plot_shap_summary(clf, X, feature_names, max_display=20, title="SHAP Feature Importance"):
        explainer = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(X)
        sv = shap_values[1] if isinstance(shap_values, list) else shap_values
        shap.summary_plot(sv, X, feature_names=feature_names,
                          max_display=max_display, show=False)
        plt.title(title)
        plt.tight_layout()
        plt.savefig(DIRS["reports"] / f"shap_{title.replace(' ','_')}.png", dpi=150)
        plt.show()
        print(f"SHAP plot saved → {DIRS['reports']}")

    print("XGBoost + LightGBM + SHAP ready.")
    print("Usage:")
    print("  X, y, feat_cols = prepare_tabular_data(feature_matrix)")
    print("  xgb_clf = train_xgboost(X_train, y_train, X_val, y_val)")
    print("  lgb_clf = train_lightgbm(X_train, y_train, X_val, y_val)")
    print("  proba   = ensemble_predict_proba(xgb_clf, lgb_clf, X_test)")
    print("  plot_shap_summary(xgb_clf, X_val, feat_cols)")


## 15. Multi-Modal Fusion — Genomic Features + CNN Image Embeddings

When CNN image embeddings are available, we concatenate them with tabular genomic features
and pass through a fusion MLP. This architecture supports both uni-modal and multi-modal inference.


In [ ]:
if TORCH_AVAILABLE:
    class GenomicVariantFusionModel(nn.Module):
        """
        Multi-modal fusion:
          tabular branch  → MLP
          image branch    → TumorCNN embedding (512-d)
          fusion head     → concatenated → MLP → classification
        """
        def __init__(self, tabular_dim: int, image_emb_dim: int = 512,
                     hidden_dim: int = 256, num_classes: int = 2,
                     use_image: bool = True):
            super().__init__()
            self.use_image = use_image

            # Tabular branch
            self.tabular_branch = nn.Sequential(
                nn.Linear(tabular_dim, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, 128),
                nn.ReLU(),
            )

            # Fusion head
            fusion_in = 128 + (image_emb_dim if use_image else 0)
            self.fusion_head = nn.Sequential(
                nn.Linear(fusion_in, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.25),
                nn.Linear(hidden_dim, num_classes),
            )

        def forward(self, tabular: torch.Tensor,
                    image_emb: torch.Tensor = None) -> torch.Tensor:
            tab_out = self.tabular_branch(tabular)
            if self.use_image and image_emb is not None:
                fused = torch.cat([tab_out, image_emb], dim=1)
            else:
                fused = tab_out
            return self.fusion_head(fused)


    # ── Demo instantiation ────────────────────────────────────────────────────
    N_TABULAR_FEATURES = 30   # update after feature matrix is built
    fusion_model = GenomicVariantFusionModel(
        tabular_dim   = N_TABULAR_FEATURES,
        image_emb_dim = 512,
        num_classes   = 2,
        use_image     = True,
    )
    print("Fusion model architecture:")
    print(fusion_model)
    print(f"\nFusion model trainable params: {sum(p.numel() for p in fusion_model.parameters() if p.requires_grad):,}")


## 16. Distributed Processing — Apache Spark Pipeline

For production-scale VCF processing across millions of variants, we wrap the feature
engineering pipeline in Spark. The same logic runs on a single node or a cluster unchanged.


In [ ]:
try:
    from pyspark.sql import SparkSession
    import pyspark.sql.functions as F
    from pyspark.sql.types import (StructType, StructField, StringType,
                                   IntegerType, DoubleType)
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    print("PySpark not available. Install: pip install pyspark")

if SPARK_AVAILABLE:
    spark = (SparkSession.builder
             .appName("GenomicVariantClassifier")
             .config("spark.driver.memory",  "4g")
             .config("spark.executor.memory","4g")
             .config("spark.sql.adaptive.enabled", "true")
             .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
             .config("spark.serializer","org.apache.spark.serializer.KryoSerializer")
             .getOrCreate())
    spark.sparkContext.setLogLevel("WARN")
    print(f"Spark {spark.version} session started.")

    # ── VCF schema ────────────────────────────────────────────────────────────
    VCF_SCHEMA = StructType([
        StructField("chrom",      StringType(),  True),
        StructField("pos",        IntegerType(), True),
        StructField("id",         StringType(),  True),
        StructField("ref",        StringType(),  True),
        StructField("alt",        StringType(),  True),
        StructField("qual",       DoubleType(),  True),
        StructField("filter",     StringType(),  True),
        StructField("info",       StringType(),  True),
    ])

    def parse_vcf_to_spark(vcf_path: str):
        """Read a VCF file into a Spark DataFrame (skips ## header lines)."""
        return (spark.read
                .option("sep","\t")
                .option("comment","#")
                .schema(VCF_SCHEMA)
                .csv(vcf_path))

    def spark_add_gnomad_af(variants_df, gnomad_parquet_path: str):
        """
        Join variant Spark DF with a gnomAD Parquet file on chrom+pos+ref+alt.
        gnomAD bulk Parquet can be generated from the gnomAD VCF using Hail or
        downloaded from the gnomAD downloads page.
        """
        gnomad_sdf = spark.read.parquet(gnomad_parquet_path)
        return variants_df.join(
            gnomad_sdf.select("chrom","pos","ref","alt","exome_af","genome_af","is_rare"),
            on=["chrom","pos","ref","alt"],
            how="left"
        )

    def spark_add_clinvar_labels(variants_df, clinvar_vcf_path: str):
        """Join with ClinVar VCF parsed as Parquet for sig_numeric label."""
        cv_sdf = spark.read.parquet(clinvar_vcf_path)
        return variants_df.join(cv_sdf.select("chrom","pos","ref","alt","sig_numeric"),
                                on=["chrom","pos","ref","alt"], how="left")

    def spark_feature_pipeline(vcf_path: str,
                                gnomad_parquet: str = None,
                                clinvar_parquet: str = None,
                                output_path: str = None) -> "DataFrame":
        """
        End-to-end Spark feature pipeline:
          1. Parse VCF
          2. Join gnomAD allele frequencies
          3. Join ClinVar labels
          4. Derive features
          5. Write to Parquet
        """
        sdf = parse_vcf_to_spark(vcf_path)

        if gnomad_parquet:
            sdf = spark_add_gnomad_af(sdf, gnomad_parquet)

        if clinvar_parquet:
            sdf = spark_add_clinvar_labels(sdf, clinvar_parquet)

        # Derived features
        sdf = (sdf
               .withColumn("is_snv",    F.when(
                   (F.length("ref") == 1) & (F.length("alt") == 1), 1).otherwise(0))
               .withColumn("is_indel",  F.when(
                   (F.length("ref") != 1) | (F.length("alt") != 1), 1).otherwise(0))
               .withColumn("log10_af",
                   F.when(F.col("exome_af") > 0,
                          F.log(F.lit(10), F.col("exome_af"))).otherwise(-8.0))
               )

        if output_path:
            (sdf.write.mode("overwrite")
                .parquet(output_path))
            print(f"Spark features written → {output_path}")
        return sdf

    print("\nSpark pipeline functions ready.")
    print("Usage:")
    print("  sdf = spark_feature_pipeline(")
    print("      vcf_path='variants.vcf',")
    print("      gnomad_parquet='gnomad_v4.parquet',")
    print("      clinvar_parquet='clinvar.parquet',")
    print("      output_path='output/features'")
    print("  )")


## 17. Pipeline Summary & Next Steps

In [ ]:
import json
from datetime import datetime

summary = {
    "stage":       "Stage 2 — Multi-Source Ingestion + Multi-Modal Modeling",
    "timestamp":   datetime.now().isoformat(),
    "databases_integrated": [
        "ClinVar (NCBI E-utilities)",
        "gnomAD v4 (GraphQL API)",
        "UniProt REST v2",
        "GWAS Catalog (EMBL-EBI REST)",
        "Open Targets Genetics (GraphQL)",
        "GTEx v8 Portal API",
        "ENCODE/SCREEN cCRE API",
        "CADD v1.7 REST API",
        "REVEL v1.3 (bulk file)",
        "AlphaMissense (bulk file)",
        "TCGA/GDC API — somatic mutations + diagnostic slides",
    ],
    "models_defined": [
        "TumorCNN (ResNet-50 fine-tuned, 512-d embedding)",
        "XGBoost tabular classifier",
        "LightGBM tabular classifier",
        "Soft-vote ensemble",
        "GenomicVariantFusionModel (multi-modal MLP)",
    ],
    "infrastructure": ["Apache Spark distributed VCF pipeline"],
    "next_steps": [
        "1. Download REVEL and AlphaMissense bulk files; update REVEL_PATH / AM_PATH",
        "2. Extract TCGA diagnostic slide tiles using openslide-python for CNN training",
        "3. Scale variant queries to genome-wide using Spark + gnomAD/ClinVar Parquet",
        "4. Run cross-validated XGBoost/LightGBM on full ClinVar label set",
        "5. Fine-tune TumorCNN on TCGA slide tiles (start with TCGA-BRCA)",
        "6. Train fusion model end-to-end once CNN embeddings are ready",
        "7. Evaluate with AUROC, AUPRC, calibration curves, SHAP explanations",
        "8. Package as REST API with FastAPI + Docker for portfolio deployment",
    ],
}

print(json.dumps(summary, indent=2))

report_path = DIRS["reports"] / "stage2_summary.json"
with open(report_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved → {report_path}")


In [ ]:
[New Markdown cell]
## Mi X — Feature Engineering
(where X = last existing section number + 1)